In [10]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
import torch

In [ ]:
!unzip mbart_lora_es_pt.zip -d mbart_lora_es_pt

Archive:  mbart_lora_es_pt.zip
replace mbart_lora_es_pt/__MACOSX/._mbart_lora_es_pt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: mbart_lora_es_pt/__MACOSX/._mbart_lora_es_pt  
replace mbart_lora_es_pt/mbart_lora_es_pt/adapter_model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/adapter_model.safetensors  
replace mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._adapter_model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._adapter_model.safetensors  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/tokenizer_config.json  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._tokenizer_config.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/special_tokens_map.json  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._special_tokens_map.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/sentencepiece.bpe.model  
  inflating: mbart_lora_es_pt/__MACOSX/mba

In [11]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
base_model = MBartForConditionalGeneration.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, "./mbart_lora_es_pt/mbart_lora_es_pt")

In [12]:
model.train()
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927


In [13]:
src_lang = "es_XX"
tgt_lang = "pt_XX"
tokenizer.src_lang = src_lang

# Función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Indicamos que la secuencia de salida debe comenzar en portugués
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
        max_length=128
    )

    traduccion = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return traduccion

In [14]:
frases = [
    "Hola mundo",
    "¿Cómo estás?",
    "Me gusta aprender cosas nuevas.",
    "Estoy entrenando un modelo de traducción.",
    "La inteligencia artificial es fascinante."
]

for f in frases:
    print(f"ES: {f}")
    print(f"PT: {traducir(f)}\n")

ES: Hola mundo
PT: Olá à vossa gente!

ES: ¿Cómo estás?
PT: Como é que estás?

ES: Me gusta aprender cosas nuevas.
PT: Eu gosto de aprender coisas novas.

ES: Estoy entrenando un modelo de traducción.
PT: Estou fazendo um modelo de tradução.

ES: La inteligencia artificial es fascinante.
PT: A inteligência artificial é fascinante.



In [ ]:
!pip install -U datasets

In [ ]:
import huggingface_hub
huggingface_hub.login() # now you will be prompted to enter your token; enter it.

In [15]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="dev")
ds_lit = load_dataset("openlanguagedata/flores_plus", "lit_Latn", split="dev")
parallel_lit = [{"translation": {"es": e["text"], "lit": g["text"]}} for e, g in zip(ds_es, ds_lit)]

dataset_lit = Dataset.from_list(parallel_lit).train_test_split(test_size=0.1, seed=42)
train_lit = dataset_lit["train"]
eval_lit = dataset_lit["test"]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

lit_Latn.parquet:   0%|          | 0.00/123k [00:00<?, ?B/s]

lit_Latn.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [16]:
tokenizer.src_lang = "es_XX"
tokenizer.tgt_lang = "lt_LT"

def tokenize_gl(batch):
    src = [x["es"] for x in batch["translation"]]
    tgt = [x["lit"] for x in batch["translation"]]
    inputs = tokenizer(src, truncation=True, padding="max_length", max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(tgt, truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized_lit = train_lit.map(tokenize_gl, batched=True)
eval_tokenized_lit = eval_lit.map(tokenize_gl, batched=True)

Map:   0%|          | 0/897 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
training_args_lit = Seq2SeqTrainingArguments(
    output_dir="./mbart_lora_es_lit",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_lit",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    label_names=["labels"]
)

trainer_lit = Seq2SeqTrainer(
    model=model,
    args=training_args_lit,
    train_dataset=train_tokenized_lit,
    eval_dataset=eval_tokenized_lit,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [21]:
trainer_lit.train()
model.save_pretrained("./mbart_lora_es_lit")
tokenizer.save_pretrained("./mbart_lora_es_lit")

Epoch,Training Loss,Validation Loss
1,No log,8.138442
2,No log,8.150839
3,8.025700,8.166649
4,8.025700,8.171156
5,7.898500,8.191551
6,7.898500,8.213914
7,7.844700,8.230454
8,7.844700,8.240876
9,7.780300,8.259391
10,7.780300,8.267627


('./mbart_lora_es_lit/tokenizer_config.json',
 './mbart_lora_es_lit/special_tokens_map.json',
 './mbart_lora_es_lit/sentencepiece.bpe.model',
 './mbart_lora_es_lit/added_tokens.json',
 './mbart_lora_es_lit/tokenizer.json')

In [22]:
pip install evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 14.6 MB/s eta 0:00:00


In [23]:
from evaluate import load

In [24]:
def test_translation_batch(sentences_es, references_gl, model_path="./mbart_lora_es_lit"):
    try:
        # Cargar modelo base + adaptadores LoRA
        base_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
        model = PeftModel.from_pretrained(base_model, model_path)
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        # Cargar tokenizer
        tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
        inputs = tokenizer(sentences_es, return_tensors="pt", padding=True, truncation=True).to(device)

        # Generar traducciones
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        translations = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]

        # Mostrar resultados
        for src, pred, ref in zip(sentences_es, translations, references_gl):
            print(f"ES: {src}")
            print(f"LIT (pred): {pred}")
            print(f"LIT (ref):  {ref}")
            print("-" * 60)

        # Calcular métricas BLEU y chrF en lote
        print("\n📊 Métricas globales:")
        bleu = load("bleu")
        chrf = load("chrf")

        bleu_score = bleu.compute(predictions=translations, references=[[r] for r in references_gl])
        chrf_score = chrf.compute(predictions=translations, references=references_gl)

        print(f"BLEU: {bleu_score['bleu']:.4f}")
        print(f"chrF: {chrf_score['score']:.2f}")

    except Exception as e:
        print(f"❌ Error en traducción: {e}")

In [25]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest[:50]")
ds_lit = load_dataset("openlanguagedata/flores_plus", "lit_Latn", split="devtest[:50]")

# Tomar una muestra de 50 ejemplos para visualización
sample_es = ds_es["text"][:50]
sample_lit = ds_lit["text"][:50]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

In [26]:
test_translation_batch(sample_es, sample_lit)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ES: «Actualmente, tenemos ratones de cuatro meses de edad que antes solían ser diabéticos y que ya no lo son», agregó.
LIT (pred): „Šiuo metu turime keturių mėnesių sergančius šuolius, kurie anksčiau būdavo diabetais ir dabar to nebėra“, – pridūrė V. M. Čiurlionis.
LIT (ref):  „Turime 4 mėnesių amžiaus peles, kurios sirgo diabetu, o dabar neserga“, – pridūrė jis.
------------------------------------------------------------
ES: La investigación todavía se ubica en su etapa inicial, conforme indicara el Dr. Ehud Ur, docente en la carrera de medicina de la Universidad de Dalhousie, en Halifax, Nueva Escocia, y director del departamento clínico y científico de la Asociación Canadiense de Diabetes.
LIT (pred): Dr. Ehud Ur, Dalhosie universiteto Halifake, Naujojoje Escocijoje profesorius, gydytojo karjeros, ir Kanados diabeto asociacijos gydytojos ir mokslo skyriaus direktoriaus, teigia, kad tyrimai dar yra pradiniame etape.
LIT (ref):  Halifakse (Naujoji Škotija) įsikūrusio Dalhauzio univer

BLEU: 0.1213
chrF: 44.09


In [27]:
import shutil
from google.colab import files

# Comprimir la carpeta (reemplaza 'nombre_de_tu_carpeta')
shutil.make_archive('mbart_lora_es_lit', 'zip', 'mbart_lora_es_lit')

# Descargar el archivo ZIP
files.download('mbart_lora_es_lit.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>